In [2]:
# Setup (Set working directory, paths, and random seeds)
# Load modules
import os                    # working directory 
import subprocess            # git clone
import sys                   # check Colab
from pathlib import Path     # paths

# Set working directory (w/ git clone)
IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()

if IN_COLAB:
    ROOT = Path('/content/dl_practice')
    if not ROOT.exists(): 
        subprocess.run(['git', 'clone', 'https://github.com/seungyong0223/dl_practice', str(ROOT) ])
    os.chdir(ROOT)
else:
    if '__file__' in dir(): 
        ROOT = Path(__file__).resolve().parents[1]
    else: 
        ROOT = Path.cwd()
    os.chdir(ROOT)
    
print(f"Working directory is {ROOT}")

# Define paths 
DATA_DIR = ROOT / "data"
CKPT_DIR = ROOT / "checkpoints"
OUTPUT_DIR = ROOT / "outputs"

for d in [DATA_DIR, CKPT_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    
# Set random seeds 
SEED = 1

import random
import numpy as np 
import torch 

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Check CPU/GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Current Device is {device}")


Working directory is /content/dl_practice
Current Device is cuda


In [3]:
# Load modules 
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

# Domain specific libraries 'TorchText', 'TorchVision', and 'TorchAudio'
# All of them include own datasets

# There are two types of datasets: 
# (1) Map-stype datasets and (2) iterable-stype datasets ...

In [4]:
# Download training data 
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

# Download test data
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 167kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.11MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 17.8MB/s]


In [5]:
# Load data 
# DataLoader supports: 
# automatic batching, sampling, shuffling, and multiprocess data loading

# Set batch size
batch_size = 64

# Create data loaders 
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader: 
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

# NCHW convention 
# N - number of samples
# C - channel (grayscale=1, RGB=3)
# H - height
# W - width

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [6]:
# Create Models

# Define model 
class NeuralNetwork(nn.Module):      # inherit & define class 
    def __init__(self): 
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512), 
            nn.ReLU(), 
            nn.Linear(512, 512), 
            nn.ReLU(), 
            nn.Linear(512, 10)
        )
    
    def forward(self, x): 
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)    # to GPU
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [7]:
# Optimization 
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [8]:
# Training loop 
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        
        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y) 
        
        # Backprobagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        if batch % 100 == 0: 
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

In [9]:
# Test Loop
def test(dataloader, model, loss_fn): 
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad(): 
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [10]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.294026 [   64/60000]
loss: 2.285374 [ 6464/60000]
loss: 2.261673 [12864/60000]
loss: 2.258286 [19264/60000]
loss: 2.246893 [25664/60000]
loss: 2.204551 [32064/60000]
loss: 2.224776 [38464/60000]
loss: 2.186472 [44864/60000]
loss: 2.188019 [51264/60000]
loss: 2.143335 [57664/60000]
Test Error: 
 Accuracy: 34.9%, Avg loss: 2.144164 

Epoch 2
-------------------------------
loss: 2.156179 [   64/60000]
loss: 2.144095 [ 6464/60000]
loss: 2.082209 [12864/60000]
loss: 2.105026 [19264/60000]
loss: 2.054901 [25664/60000]
loss: 1.977896 [32064/60000]
loss: 2.022960 [38464/60000]
loss: 1.940120 [44864/60000]
loss: 1.950309 [51264/60000]
loss: 1.860950 [57664/60000]
Test Error: 
 Accuracy: 54.3%, Avg loss: 1.869543 

Epoch 3
-------------------------------
loss: 1.901327 [   64/60000]
loss: 1.868703 [ 6464/60000]
loss: 1.750409 [12864/60000]
loss: 1.800762 [19264/60000]
loss: 1.687900 [25664/60000]
loss: 1.629443 [32064/60000]
loss: 1.660456 [38464/

In [11]:
# Save models 
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


In [13]:
# Load models 
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [14]:
# Prediction 
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad(): 
    x=x.to(device)
    pred=model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"
